In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-obp qiskit-addon-utils rustworkx
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 기댓값 추정을 위한 연산자 역전파(OBP)

*사용 시간 예상: Eagle r3 프로세서에서 16분 (참고: 이는 추정치이며 실제 실행 시간은 다를 수 있습니다.)*

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.primitives import StatevectorEstimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import LieTrotter

from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_depth, combine_slices
from qiskit_addon_obp.utils.simplify import OperatorBudget
from qiskit_addon_obp import backpropagate
from qiskit_addon_obp.utils.truncating import setup_budget

from rustworkx.visualization import graphviz_draw

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2, EstimatorOptions

## 배경
연산자 역전파(Operator backpropagation)는 양자 회로의 끝에서 연산을 측정 대상 관측 가능량(observable)에 흡수하는 기법으로, 일반적으로 관측 가능량의 항이 늘어나는 대신 회로의 깊이를 줄일 수 있습니다. 목표는 관측 가능량이 지나치게 커지지 않는 범위에서 최대한 많은 회로를 역전파하는 것입니다. Qiskit 기반 구현은 OBP Qiskit 애드온에서 제공되며, 자세한 내용은 해당 [문서](/guides/qiskit-addons-obp)와 [간단한 예제](/guides/qiskit-addons-obp-get-started)에서 확인할 수 있습니다.

관측 가능량 $O = \sum_P c_P P$를 측정해야 하는 예시 회로를 생각해봅시다. 여기서 $P$는 Pauli 연산자이고 $c_P$는 계수입니다. 회로를 단일 유니터리 $U$로 나타내면 아래 그림과 같이 $U = U_C U_Q$로 논리적으로 분할할 수 있습니다.

![Uq 다음에 Uc가 오는 회로 다이어그램](../docs/images/tutorials/improving-estimation-of-expectation-values-with-operator-backpropagation/logical-partitioning.avif)

연산자 역전파는 유니터리 $U_C$를 $O' = U_C^{\dagger}OU_C = \sum_P c_P U_C^{\dagger}PU_C$와 같이 관측 가능량에 진화시켜 흡수합니다. 즉, 계산의 일부가 관측 가능량을 $O$에서 $O'$로 진화시키는 고전적 방법으로 수행됩니다. 이제 원래 문제는 유니터리가 $U_Q$인 더 낮은 깊이의 회로에서 관측 가능량 $O'$를 측정하는 것으로 재구성할 수 있습니다.

유니터리 $U_C$는 여러 슬라이스 $U_C = U_S U_{S-1}...U_2U_1$으로 표현됩니다. 슬라이스를 정의하는 방법은 여러 가지가 있습니다. 예를 들어, 위 예시 회로에서 각 $R_{zz}$ 레이어와 각 $R_x$ 게이트 레이어를 개별 슬라이스로 볼 수 있습니다. 역전파는 $O' = \Pi_{s=1}^S \sum_P c_P U_s^{\dagger} P U_s$를 고전적으로 계산하는 과정을 포함합니다. 각 슬라이스 $U_s$는 $U_s = exp(\frac{-i\theta_s P_s}{2})$로 표현할 수 있으며, 여기서 $P_s$는 $n$-큐비트 Pauli이고 $\theta_s$는 스칼라입니다. 다음이 성립함을 쉽게 확인할 수 있습니다.

$$
U_s^{\dagger} P U_s = P \qquad \text{if} ~[P,P_s] = 0,
$$
$$
U_s^{\dagger} P U_s = \qquad cos(\theta_s)P + i sin(\theta_s)P_sP \qquad \text{if} ~{P,P_s} = 0
$$

위 예시에서 ${P,P_s} = 0$이면 기댓값을 계산하기 위해 하나가 아닌 두 개의 양자 회로를 실행해야 합니다. 따라서 역전파는 관측 가능량의 항 수를 늘려 더 많은 회로 실행이 필요할 수 있습니다. 연산자가 너무 커지지 않으면서 더 깊이 역전파하는 방법 중 하나는 연산자에 항을 추가하는 대신 계수가 작은 항을 잘라내는(truncate) 것입니다. 예를 들어, 위 예시에서 $\theta_s$가 충분히 작다면 $P_sP$를 포함하는 항을 잘라낼 수 있습니다. 항을 잘라내면 실행해야 할 양자 회로 수가 줄어들지만, 그 대가로 잘라낸 항의 계수 크기에 비례하는 오차가 최종 기댓값 계산에 발생합니다.

이 튜토리얼은 <a href="https://github.com/Qiskit/qiskit-addon-obp">qiskit-addon-obp</a>를 사용하여 하이젠베르크 스핀 체인의 양자 동역학을 시뮬레이션하는 [Qiskit 패턴](/guides/intro-to-patterns)을 구현합니다.

## 요구 사항
이 튜토리얼을 시작하기 전에 다음이 설치되어 있는지 확인하세요.

- Qiskit SDK v1.2 이상 (`pip install qiskit`)
- Qiskit Runtime v0.28 이상 (`pip install qiskit-ibm-runtime`)
- OBP Qiskit 애드온 (`pip install qiskit-addon-obp`)
- Qiskit 애드온 유틸리티 (`pip install qiskit-addon-utils`)

## 설정

In [2]:
num_qubits = 10
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)
graphviz_draw(coupling_map.graph, method="circo")

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

## 파트 I: 소규모 하이젠베르크 스핀 체인
### 1단계: 고전적 입력을 양자 문제로 매핑하기
#### 양자 하이젠베르크 모델의 시간 진화를 양자 실험으로 매핑하기
[qiskit_addon_utils](https://github.com/Qiskit/qiskit-addon-utils) 패키지는 다양한 목적으로 재사용 가능한 기능들을 제공합니다.

[qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) 모듈은 주어진 연결 그래프에서 하이젠베르크 유사 해밀토니안을 생성하는 함수를 제공합니다.
이 그래프는 [rustworkx.PyGraph](https://www.rustworkx.org/apiref/rustworkx.PyGraph.html) 또는 [CouplingMap](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.CouplingMap)일 수 있어 Qiskit 중심 워크플로에서 사용하기 편리합니다.

다음에서는 10개의 Qubit으로 이루어진 선형 체인 `CouplingMap`을 생성합니다.

In [3]:
# Get a qubit operator describing the Heisenberg XYZ model
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII', 'IIIIIIIIIX', 'IIIIIIIIIY', 'IIIIIIIIIZ', 'IIIIIIIIXI', 'IIIIIIIIYI', 'IIIIIIIIZI', 'IIIIIIIXII', 'IIIIIIIYII', 'IIIIIIIZII', 'IIIIIIXIII', 'IIIIIIYIII', 'IIIIIIZIII', 'IIIIIXIIII', 'IIIIIYIIII', 'IIIIIZIIII', 'IIIIXIIIII', 'IIIIYIIIII', 'IIIIZIIIII', 'IIIXIIIIII', 'IIIYIIIIII', 'IIIZIIIIII', 'IIXIIIIIII', 'IIYIIIIIII', 'IIZIIIIIII', 'IXIIIIIIII', 'IYIIIIIIII', 'IZIIIIIIII', 'XIIIIIIIII', 'YIIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.

![이전 코드 셀의 출력](../docs/images/tutorials/operator-back-propagation/extracted-outputs/de8ce35e-a2c5-474f-adb9-88c9afb2bd76-0.avif)

다음으로, 하이젠베르크 XYZ 해밀토니안을 모델링하는 Pauli 연산자를 생성합니다.

$$
{\hat{\mathcal{H}}_{XYZ} = \sum_{(j,k)\in E} (J_{x} \sigma_j^{x} \sigma_{k}^{x} + J_{y} \sigma_j^{y} \sigma_{k}^{y} + J_{z} \sigma_j^{z} \sigma_{k}^{z}) + \sum_{j\in V} (h_{x} \sigma_j^{x} + h_{y} \sigma_j^{y} + h_{z} \sigma_j^{z})}
$$

여기서 $G(V,E)$는 제공된 커플링 맵의 그래프입니다.

In [4]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=2),
)
circuit.draw("mpl", style="iqp", fold=-1)

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum hardware execution

#### Create circuit slices to backpropagate

The `backpropagate` function backpropagates entire circuit slices at a time. Therefore, the choice of slicing can have an impact on how well backpropagation performs for a given problem. Here, we will group gates of the same type into slices using the [`slice_by_depth`](/docs/api/qiskit-addon-utils/slicing#slice_by_depth) function.

For a more detailed discussion on circuit slicing, check out this [how-to guide](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) of the [`qiskit-addon-utils`](https://github.com/Qiskit/qiskit-addon-utils) package.

In [5]:
slices = slice_by_depth(circuit, max_slice_depth=1)
print(f"Separated the circuit into {len(slices)} slices.")

Separated the circuit into 18 slices.


큐비트 연산자로부터 시간 진화를 모델링하는 양자 회로를 생성할 수 있습니다.
다시 한번, [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) 모듈이 이를 위한 편리한 함수를 제공합니다.

In [6]:
op_budget = OperatorBudget(max_qwc_groups=8)

![이전 코드 셀의 출력](../docs/images/tutorials/operator-back-propagation/extracted-outputs/1d68f197-ffa4-49de-9fe8-243b1facbd00-0.avif)

### 2단계: 양자 하드웨어 실행을 위한 문제 최적화
#### 역전파할 회로 슬라이스 생성하기
``backpropagate`` 함수는 한 번에 전체 회로 슬라이스를 역전파한다는 점을 기억하세요. 따라서 슬라이싱 방법의 선택은 주어진 문제에서 역전파 성능에 영향을 줄 수 있습니다. 여기서는 [slice_by_gate_types](https://docs.quantum.ibm.com/api/qiskit-addon-utils/slicing#slice_by_gate_types) 함수를 사용하여 같은 유형의 게이트를 슬라이스로 묶겠습니다.

회로 슬라이싱에 대한 더 자세한 논의는 [qiskit-addon-utils](https://github.com/Qiskit/qiskit-addon-utils) 패키지의 [사용 방법 가이드](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html)를 참고하세요.

In [7]:
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits=num_qubits,
)
observable

SparsePauliOp(['IIIIIIIIIZ', 'IIIIIIIIZI', 'IIIIIIIZII', 'IIIIIIZIII', 'IIIIIZIIII', 'IIIIZIIIII', 'IIIZIIIIII', 'IIZIIIIIII', 'IZIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j,
 0.1+0.j, 0.1+0.j])

Below you will see that we backpropagated six slices, and the terms were combined into six and not eight groups. This implies that backpropagating one more slice would cause the number of Pauli groups to exceed eight. We can verify that this is the case by inspecting the returned metadata. Also note that in this portion the circuit transformation is exact.  That is, no terms of the new observable $O’$ were truncated. The backpropagated circuit and the backpropagated operator give the exact outcome as the original circuit and operator.

In [8]:
# Backpropagate slices onto the observable
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
# Recombine the slices remaining after backpropagation
bp_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs.paulis)} terms, which can be combined into {len(bp_obs.group_commuting(qubit_wise=True))} groups."
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit.draw("mpl", fold=-1, scale=0.6)

Backpropagated 6 slices.
New observable has 60 terms, which can be combined into 6 groups.
Note that backpropagating one more slice would result in 114 terms across 12 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/ee8fd385-1.avif" alt="Output of the previous code cell" />

#### 역전파 중 연산자 증가 크기 제한하기
역전파 중 연산자의 항 수는 일반적으로 $N$이 Qubit 수일 때 $4^N$에 빠르게 근접합니다. 두 항이 큐비트 단위로 교환되지 않으면(non-commuting), 해당 항들의 기댓값을 얻기 위해 별도의 회로가 필요합니다. 예를 들어, 2-큐비트 관측 가능량 $O = 0.1 XX + 0.3 IZ - 0.5 IX$가 있다면, $[XX,IX] = 0$이므로 이 두 항의 기댓값을 계산하는 데 단일 기저 측정으로 충분합니다. 하지만 $IZ$는 나머지 두 항과 반교환(anti-commute)합니다. 따라서 $IZ$의 기댓값을 계산하려면 별도의 기저 측정이 필요합니다. 즉, $\langle O \rangle$을 계산하는 데 하나가 아닌 두 개의 회로가 필요합니다. 연산자의 항 수가 증가할수록 필요한 회로 실행 횟수도 증가할 가능성이 있습니다.

연산자의 크기는 ``backpropagate`` 함수의 ``operator_budget`` 키워드 인수를 지정하여 제한할 수 있으며, 이 인수는 [OperatorBudget](https://docs.quantum.ibm.com/api/qiskit-addon-obp/utils-simplify#operatorbudget) 인스턴스를 받습니다.

추가 리소스(시간) 할당량을 제어하기 위해, 역전파된 관측 가능량이 가질 수 있는 큐비트 단위 교환 Pauli 그룹의 최대 수를 제한합니다. 여기서는 연산자의 큐비트 단위 교환 Pauli 그룹 수가 8을 초과하면 역전파를 중단하도록 지정합니다.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(backend)

<IBMBackend('ibm_kingston')>


In [10]:
pm_basis = generate_preset_pass_manager(
    optimization_level=3, basis_gates=backend.configuration().basis_gates
)
isa_circuit = pm_basis.run(circuit)
isa_bp_circuit = pm_basis.run(bp_circuit)

### Step 3: Execute using Qiskit primitives

First, we create two [Primitive Unified Blocs](/docs/api/qiskit/primitives) (PUBs) corresponding to the original circuit, and the backpropagated circuit. Then we execute the pubs on an ideal Estimator to obtain the expectation values.

In [11]:
pubs = [(isa_circuit, observable), (isa_bp_circuit, bp_obs)]

In [12]:
rng = np.random.default_rng()
estimator = StatevectorEstimator(seed=rng)
job = estimator.run(pubs)

### Step 4: Post-process and return result in desired classical format

Now we obtain the expectation values of the original and backpropagated circuits.

In [13]:
primitive_result = job.result()
circuit_expval = primitive_result[0].data.evs.item()
bp_circuit_expval = primitive_result[1].data.evs.item()

In [14]:
methods = [
    "No backpropagation",
    "Backpropagation",
]
values = [circuit_expval, bp_circuit_expval]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylim([0.6, 0.92])
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

슬라이스당 `5e-3`의 오차를 잘라내기에 할당함으로써, 원래의 여덟 개 교환 Pauli 그룹 예산 내에서 회로에서 슬라이스를 하나 더 제거할 수 있습니다. 기본적으로 `backpropagate`는 잘라낸 계수의 L1 노름을 사용하여 잘라내기로 인한 총 오차를 제한합니다. 다른 옵션은 [p_norm 지정 사용 방법 가이드](https://qiskit.github.io/qiskit-addon-obp/how_tos/bound_error_using_p_norm.html)를 참고하세요.

7개의 슬라이스를 역전파한 이 예시에서 총 잘라내기 오차는 ``(슬라이스당 5e-3 오차) * (7 슬라이스) = 3.5e-2``를 초과하지 않아야 합니다.
슬라이스에 오차 예산을 분배하는 방법에 대한 자세한 논의는 [이 사용 방법 가이드](https://qiskit.github.io/qiskit-addon-obp/how_tos/truncate_operator_terms.html)를 참고하세요.

In [15]:
num_qubits = 50
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)

hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)

# Generate a time evolution circuit for the Hamiltonian
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=4),
)

# Define the observable to measure
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits,
)

slices = slice_by_depth(circuit, max_slice_depth=1)

# Define the maximum number of qwc groups allowed in the backpropagated observable, and the truncation error budget
op_budget = OperatorBudget(max_qwc_groups=15)
truncation_error_budget = setup_budget(
    max_error_total=0.03, max_error_per_slice=0.005
)

# First backpropagation without truncation
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
bp_circuit = combine_slices(remaining_slices)

# Now backpropagate with truncation, using the same operator budget and the defined truncation error budget
bp_obs_trunc, remaining_slices_trunc, metadata = backpropagate(
    observable,
    slices,
    operator_budget=op_budget,
    truncation_error_budget=truncation_error_budget,
)
bp_circuit_trunc = combine_slices(
    remaining_slices_trunc, include_barriers=False
)

# Now we transpile the original circuit and the two backpropagated circuits, and apply the layout to the corresponding observables
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

isa_circuit = pm.run(circuit)
isa_bp_circuit = pm.run(bp_circuit)
isa_bp_circuit_trunc = pm.run(bp_circuit_trunc)

isa_observable = observable.apply_layout(isa_circuit.layout)
isa_bp_observable = bp_obs.apply_layout(isa_bp_circuit.layout)
isa_bp_observable_trunc = bp_obs_trunc.apply_layout(
    isa_bp_circuit_trunc.layout
)

# Compare the 2-qubit depth of each transpiled circuit to see how much depth backpropagation saved
print(
    f"2-qubit depth without backpropagation: {isa_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation: {isa_bp_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation and truncation: {isa_bp_circuit_trunc.depth(lambda x: x.operation.num_qubits == 2)}"
)

pubs = [
    (isa_circuit, isa_observable),
    (isa_bp_circuit, isa_bp_observable),
    (isa_bp_circuit_trunc, isa_bp_observable_trunc),
]

# Now we instantiate the Estimator primitive for the hardware with ZNE and measurement error mitigation
# and compute the three circuits and observables
options = EstimatorOptions()
options.default_precision = 0.01
options.resilience_level = 2
options.resilience.zne.noise_factors = [1, 1.2, 1.4]
options.resilience.zne.extrapolator = ["linear"]
estimator = EstimatorV2(mode=backend, options=options)

estimator.options.environment.job_tags = ["TUT_OBP"]
job = estimator.run(pubs)

# Retrieve the results and the standard deviations
result_no_bp = job.result()[0].data.evs.item()
result_bp = job.result()[1].data.evs.item()
result_bp_trunc = job.result()[2].data.evs.item()

std_no_bp = job.result()[0].data.stds.item()
std_bp = job.result()[1].data.stds.item()
std_bp_trunc = job.result()[2].data.stds.item()

2-qubit depth without backpropagation: 24
2-qubit depth with backpropagation: 20
2-qubit depth with backpropagation and truncation: 18


In [16]:
print(f"Expectation value without backpropagation: {result_no_bp}")
print(f"Backpropagated expectation value: {result_bp}")
print(f"Backpropagated expectation value with truncation: {result_bp_trunc}")

Expectation value without backpropagation: 0.9543907942381811
Backpropagated expectation value: 0.9445337385406468
Backpropagated expectation value with truncation: 0.934050286970965


In [17]:
# Plot the results
methods = [
    "No backpropagation",
    "Backpropagation",
    "Backpropagation w/ truncation",
]
values = [result_no_bp, result_bp, result_bp_trunc]
error_bars = [std_no_bp, std_bp, std_bp_trunc]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.errorbar(methods, values, yerr=error_bars, fmt="o", color="r", capsize=5)
plt.axhline(0.89)
ax.set_ylim([0.8, 0.98])
plt.text(0.25, 0.895, "Exact result")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/37834c72-1.avif" alt="Output of the previous code cell" />

## Next steps
If you found this work interesting, you might be interested in the following material:
<Admonition type="tip" title="Recommendations">
- [Approximate quantum compilation for time evolution circuits](/docs/tutorials/approximate-quantum-compilation-for-time-evolution)
- [Multi-product formulas to reduce Trotter error](/docs/tutorials/multi-product-formula)
- [`pauli-prop`](https://github.com/Qiskit/pauli-prop), a Rust-accelerated package for Pauli propagation, with [tutorials](https://github.com/Qiskit/pauli-prop/tree/main/docs/tutorials) covering OBP, classical expectation-value estimation, and noisy simulation
</Admonition>